# Instalacion

Los siguientes comandos instalan a traves de pip3 las dependencias requeridas por el Notebook

In [368]:
!pip3 install pandas
!pip3 install openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


# Lectura del Corpus

En esta funcion, se realiza una lectura del corpus, para poder genera la cantidad de frases que contiene el archivo

In [369]:
from collections import defaultdict

def leer_archivo(nombre_archivo):
    with open(nombre_archivo, 'r', encoding='utf8') as archivo:
        palabras = []
        palabra_actual = []

        for linea_actual in archivo:
            linea_actual = linea_actual.strip()

            if linea_actual.startswith('<doc') or linea_actual == '':
                continue

            columnas = linea_actual.split()
            if len(columnas) < 3:
                continue

            # Extraemos palabra y etiqueta
            palabra = columnas[0].lower() # Normalización a minúsculas
            etiqueta = columnas[2]
            
            palabra_actual.append((palabra, etiqueta))

            # Si la etiqueta es un punto final (Fp), cerramos la frase
            if etiqueta == 'Fp':
                palabras.append(palabra_actual)
                palabra_actual = []

        # Por si el archivo no termina en punto, guardamos lo último
        if palabra_actual:
            palabras.append(palabra_actual)

    return palabras

palabras = leer_archivo('corpus-tagged.txt')

[[('tristana', 'NP00000'), ('es', 'VSIP3S0'), ('una', 'DI0FS0'), ('película', 'NCFS000'), ('de', 'SPS00'), ('el', 'DA0MS0'), ('director', 'NCMS000'), ('español', 'AQ0MS0'), ('nacionalizado', 'VMP00SM'), ('mexicano', 'AQ0MS0'), ('luis_buñuel', 'NP00000'), ('.', 'Fp')], [('está', 'VAIP3S0'), ('basada', 'VMP00SF'), ('en', 'SPS00'), ('la', 'DA0FS0'), ('novela', 'NCFS000'), ('de', 'SPS00'), ('el', 'DA0MS0'), ('mismo', 'AQ0MS0'), ('nombre', 'NCMS000'), ('de', 'SPS00'), ('benito_pérez_galdós', 'NP00000'), ('.', 'Fp')], [('fue', 'VSIS3S0'), ('nominada', 'VMP00SF'), ('a', 'SPS00'), ('el', 'DA0MS0'), ('oscar', 'NP00000'), ('a', 'SPS00'), ('la', 'DA0FS0'), ('mejor', 'AQ0CS0'), ('película', 'NCFS000'), ('de', 'SPS00'), ('habla', 'NCFS000'), ('no', 'RN'), ('inglesa', 'AQ0FS0'), ('en', 'SPS00'), ('1970', 'Z'), ('.', 'Fp')], [('tristana', 'NP00000'), ('y', 'CC'), ('nazarín', 'NP00000'), ('son', 'VSIP3P0'), ('las', 'DA0FP0'), ('dos', 'Z'), ('novelas', 'NCFP000'), ('de', 'SPS00'), ('benito_pérez_galdós

# Obtencion de etiquetas, emisiones y transiciones

En esta funcion, se obtienen todas las etiquetas, emisiones y transiciones que existen en el corpus

In [370]:
def obtener_probabilidades():
    etiquetas = defaultdict(int)
    emisiones = defaultdict(int)
    transiciones = defaultdict(int)

    for palabra_etiqueta in palabras:
    
        etiqueta_anterior = '<START>'
    
        for palabra, etiqueta in palabra_etiqueta:
            etiquetas[etiqueta] += 1
            emisiones[(palabra, etiqueta)] += 1
            transiciones[(etiqueta_anterior, etiqueta)] += 1
    
            etiqueta_anterior = etiqueta
    
        transiciones[(etiqueta_anterior, '<END>')] += 1

    return etiquetas, emisiones, transiciones

etiquetas, emisiones, transiciones = obtener_probabilidades()

ETIQUETAS defaultdict(<class 'int'>, {'NP00000': 320, 'VSIP3S0': 31, 'DI0FS0': 46, 'NCFS000': 271, 'SPS00': 683, 'DA0MS0': 157, 'NCMS000': 273, 'AQ0MS0': 41, 'VMP00SM': 76, 'Fp': 178, 'VAIP3S0': 15, 'VMP00SF': 31, 'DA0FS0': 142, 'VSIS3S0': 21, 'AQ0CS0': 73, 'RN': 18, 'AQ0FS0': 35, 'Z': 55, 'CC': 152, 'VSIP3P0': 3, 'DA0FP0': 18, 'NCFP000': 49, 'CS': 99, 'VMIS3S0': 69, 'VSN0000': 12, 'PI0MS000': 3, 'DD0MP0': 5, 'NCMP000': 128, 'RG': 127, 'VMP00PM': 13, 'DI0FP0': 8, 'VMN0000': 88, 'PP3FSA00': 4, 'Fd': 17, 'Fc': 206, 'PI0FS000': 2, 'VASI1S0': 1, 'VAP00SM': 2, 'AO0FS0': 12, 'DP3CP0': 18, 'DI0MS0': 65, 'PR0FS000': 1, 'DP3CS0': 61, 'PD0MS000': 4, 'VMIP3S0': 134, 'VMG0000': 32, 'PP3CNA00': 15, 'VSIF3S0': 1, 'DD0MS0': 13, 'P0000000': 69, 'W': 11, 'NCMN000': 13, 'DA0MP0': 44, 'NCCP000': 14, 'VMIP3P0': 41, 'AQ0FP0': 11, 'PR0CN000': 78, 'PP3MPA00': 4, 'DA0NS0': 15, 'AQ0CP0': 10, 'AQ0MP0': 15, 'PR0CS000': 17, 'DI0MP0': 18, 'Fe': 54, 'VMSP3P0': 4, 'PP3CN000': 8, 'Fpa': 31, 'Fpt': 31, 'VMIS3P0': 4, '

# Obtencion de las probabilidades de emision

En esta funcion, se obtienen las probabilidades de las emisiones

In [363]:
def obtener_probabilidades_emision():
    probabilidades = {}

    for (palabra, etiqueta), contador in emisiones.items():
        probabilidades[(palabra, etiqueta)] = contador / etiquetas[etiqueta]

    return probabilidades

probabilidades_emision = obtener_probabilidades_emision()

# Obtencion de las probabilidades de transicion

En esta funcion, se obtienen las probabilidades de las transiciones

In [364]:
def obtener_probabilidades_transicion():
    probabilidades = {}

    for (etiqueta1, etiqueta2), contador in transiciones.items():
        if etiqueta1 == '<START>':
            total = sum(valor for (e1, e2), valor in transiciones.items() if e1 == '<START>')
        else:
            total = etiquetas[etiqueta1]
    
        probabilidades[(etiqueta1, etiqueta2)] = contador / total

    return probabilidades

probabilidades_transicion = obtener_probabilidades_transicion()

# Exportar a probabilidades de emision y transicion a Excel

En esta funcion se exportan las probabilidades de emision y transicion del corpus analizado

In [371]:
import pandas as pd

def exportarEmisionExcel(): 
    emisionExcel = pd.DataFrame([
        (palabra, etiqueta, probabilidad) for (palabra, etiqueta), probabilidad in probabilidades_emision.items()
    ], columns=['palabra', 'etiqueta', 'probabilidad'])
    
    emisionExcel.to_excel('probabilidades_emision.xlsx', index=False)

def exportarTransicionExcel():
    transicionExcel = pd.DataFrame([
        (etiqueta1, etiqueta2, probabilidad) for (etiqueta1, etiqueta2), probabilidad in probabilidades_transicion.items()
    ], columns=['etiqueta anterior', 'etiqueta siguiente', 'probabilidad'])
    
    transicionExcel.to_excel('probabilidades_transicion.xlsx', index=False)

exportarEmisionExcel()
exportarTransicionExcel()

# Algoritmo de Viterbi

En esta funcion se desarrolla el algoritmo de Viterbi

In [374]:
def algoritmo_viterbi(palabras, etiquetas, probabilidades_transicion, probabilidades_emision):

    suavizador = 1e-10

    viterbi = [{}]
    ruta = {}

    for etiqueta in etiquetas:
        viterbi[0][etiqueta] = probabilidades_transicion.get(('<START>', etiqueta), suavizador) * \
                    probabilidades_emision.get((palabras[0], etiqueta), suavizador)
        ruta[etiqueta] = [etiqueta]

    for palabra in range(1, len(palabras)):
        viterbi.append({})
        nueva_ruta = {}

        for etiqueta in etiquetas:

            probabilidad, estado = max(
                (viterbi[palabra-1][etiqueta_anterior] *
                 probabilidades_transicion.get((etiqueta_anterior, etiqueta), suavizador) *
                 probabilidades_emision.get((palabras[palabra], etiqueta), suavizador), etiqueta_anterior)
                for etiqueta_anterior in etiquetas
            )

            viterbi[palabra][etiqueta] = probabilidad
            nueva_ruta[etiqueta] = ruta[estado] + [etiqueta]

        ruta = nueva_ruta

    probabilidad, estado = max(
        (viterbi[len(palabras)-1][etiqueta] *
         probabilidades_transicion.get((etiqueta, '<END>'), suavizador), etiqueta)
        for etiqueta in etiquetas
    )

    return probabilidad, ruta[estado], viterbi

# Ejecutar el algoritmo de Viterbi

En esta funcion se muestra la probabilidad de Viterbi dada una oracion

In [375]:
def mostrar_probabilidad_viterbi(oracion):
    oracion_limpia = [p.lower() for p in oracion]
    
    tags = list(etiquetas.keys())
    probabilidad, ruta, viterbi = algoritmo_viterbi(oracion_limpia, tags, probabilidades_transicion, probabilidades_emision)
    
    print('Probabilidad de Viterbi: ', probabilidad)

oracion_test = ['habla', 'con', 'el', 'enfermo', 'grave', 'de', 'transplantes', '.']
mostrar_probabilidad_viterbi(oracion_test)

Probabilidad de Viterbi:  5.873482133261211e-32
